# ROI by product: BD vs model

For each **product** we compare the return on investment (ROI):
- **Benefit** = (number of clicks) × **CPL**. The CPL is the revenue that each click yields and is specific to each product.
- **Cost**: sending emails has a cost. We analyse it with **three cost structures**, all **per email sent**:
  - **Fixed:** each email costs the same (€1), the same for all products.
  - **Variable:** each email costs a **% of the product's CPL** (20% / 30% / 40%).
  - **Mixed:** a fixed base per email **plus** a % of the CPL (€0.50 + 15% of the CPL).
- **ROI** = (benefit − cost) / cost.

We compare two ways of running the campaign:
- **BD (what there was):** the product was sent to all registered users.
- **Our model:** sends an email only if its expected benefit (`p_model × CPL`) exceeds its marginal cost, prioritising those who are likely to click.

> Missing CPL (records with no CPL reported) imputed with the median (€8).

## 1 · Data: sends, clicks and CPL per event

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
sns.set_style('whitegrid')
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p/'pyproject.toml').exists() or (p/'.git').exists():
            return p
    return Path.cwd()
PROC = find_root()/'data'/'processed'

prop = pd.read_csv(PROC/'propensity_scores.csv')
prods = pd.read_csv(PROC/'products.csv')[['id_product','cpl']]
# CPL per event (benefit per click); we impute the missing ones with the median
prop['cpl'] = pd.to_numeric(prop['id_product'].map(prods.set_index('id_product')['cpl']), errors='coerce')
prop['cpl'] = prop['cpl'].fillna(prop['cpl'].median())
prop['beneficio'] = prop['target'] * prop['cpl']            # real benefit (only if there was a click)
prop['beneficio_esperado'] = prop['p_model'] * prop['cpl']  # expected benefit (to decide whom to send to)
print('Eventos:', len(prop), '| clicks:', int(prop.target.sum()), '| CPL medio:', round(prop.cpl.mean(), 2))

## 2 · Cost parameters (the three scenarios)

All costs are **per email sent** (marginal cost):
- **Fixed:** `C_FIJO_EMAIL` € per email, the same for all products.
- **Variable:** a percentage of the product's CPL (`PCT_VAR`). We compare 20% / 30% / 40%.
- **Mixed:** a fixed base `BASE_MIX` € plus a percentage `PCT_MIX` of the CPL per email.

> Model decision rule: send if `p_model × CPL ≥ marginal_cost`. In the **variable** case this is exactly equivalent to `p_model ≥ percentage`.

In [ ]:
C_FIJO_EMAIL = 1.0              # FIXED: cost per email, the same for all products (EUR)
PCT_VAR = [0.20, 0.30, 0.40]   # VARIABLE: cost per email = % of the CPL (three scenarios to compare)
BASE_MIX, PCT_MIX = 0.50, 0.15 # MIXED: fixed base (EUR) + % of the CPL per email

def coste_marginal(escenario, cpl):
    """Cost of sending ONE email (per row), according to the cost structure and the product's CPL."""
    if escenario == 'fijo':
        return pd.Series(C_FIJO_EMAIL, index=cpl.index)
    if escenario.startswith('variable_'):
        pct = float(escenario.split('_')[1]) / 100
        return pct * cpl
    if escenario == 'mixto':
        return BASE_MIX + PCT_MIX * cpl
    raise ValueError(escenario)

ESCENARIOS = ['fijo'] + [f'variable_{int(p*100)}' for p in PCT_VAR] + ['mixto']
print('Escenarios de coste definidos:', ESCENARIOS)

## 3 · ROI by product: BD vs model, in each cost scenario

In [ ]:
def roi_por_producto(escenario):
    filas = []
    for producto, sub in prop.groupby('product_new'):
        cm = coste_marginal(escenario, sub['cpl'])   # cost per email (vector, one row per send)
        n = len(sub)
        clicks = int(sub['target'].sum())
        beneficio = sub['beneficio'].sum()
        # --- BD: sent to everyone ---
        coste_bd = cm.sum()
        roi_bd = (beneficio - coste_bd) / coste_bd if coste_bd > 0 else np.nan
        # --- Model: send an email only if its expected benefit >= its marginal cost ---
        mask = sub['beneficio_esperado'] >= cm
        n_mod = int(mask.sum())
        beneficio_mod = sub.loc[mask, 'beneficio'].sum()
        coste_mod = cm[mask].sum()
        roi_mod = (beneficio_mod - coste_mod) / coste_mod if coste_mod > 0 else np.nan
        filas.append({'producto': producto, 'cpl': round(sub['cpl'].mean(), 1),
                      'envios': n, 'clicks': clicks,
                      'ROI_BD_%': round(100 * roi_bd, 0),
                      'envios_modelo': n_mod,
                      'ROI_modelo_%': round(100 * roi_mod, 0) if not np.isnan(roi_mod) else np.nan})
    df = pd.DataFrame(filas).sort_values('envios', ascending=False).reset_index(drop=True)
    df['mejora'] = df['ROI_modelo_%'] - df['ROI_BD_%']
    return df

tablas = {}
for escenario in ESCENARIOS:
    d = roi_por_producto(escenario)
    tablas[escenario] = d
    n_mejora = int((d['mejora'] > 0).sum())
    print(f'--- COSTE {escenario.upper()}: el modelo mejora el ROI en {n_mejora}/{len(d)} productos ---')
    display(d)

In [ ]:
# --- AGGREGATE ROI per scenario (sum of benefits and costs across ALL products) ---
def roi_agregado(escenario):
    cm = coste_marginal(escenario, prop['cpl'])            # cost per email (one row per send)
    beneficio_total = prop['beneficio'].sum()
    coste_bd = cm.sum()                                     # BD: sent to everyone
    roi_bd = (beneficio_total - coste_bd) / coste_bd
    mask = prop['beneficio_esperado'] >= cm                # model: send if expected benefit >= cost
    beneficio_mod = prop.loc[mask, 'beneficio'].sum()
    coste_mod = cm[mask].sum()
    roi_mod = (beneficio_mod - coste_mod) / coste_mod if coste_mod > 0 else float('nan')
    n_mejora = int((tablas[escenario]['mejora'] > 0).sum())
    return {'escenario': escenario,
            'ROI_BD_%': round(100*roi_bd, 0),
            'ROI_modelo_%': round(100*roi_mod, 0),
            'emails_enviados_%': round(100*mask.sum()/len(prop), 1),
            'productos_mejoran': f'{n_mejora}/{len(tablas[escenario])}'}

roi_agg = pd.DataFrame([roi_agregado(e) for e in ESCENARIOS])
print('=== ROI AGREGADO por estructura de coste (modelo plano) ===')
print(roi_agg.to_string(index=False))


## 3.bis · ROI at PRODUCTION SCALE (real prior 2%)

The previous ROI is measured on the **balanced** sample (24% click rate). To estimate the **real** production ROI
we reweight each row to the real prior (2%) and decide using the **corrected** probability `p_real`
(realistic expected benefit = `p_real × CPL`). This is the economic reading at real scale.

In [ ]:
# Reweighting to the real prior: weights that take the prevalence from 0.238 (balanced) to 0.02 (production).
rho_train = float(prop['target'].mean()); rho_real = 0.02
prop['w'] = np.where(prop['target']==1, rho_real/rho_train, (1-rho_real)/(1-rho_train))
print(f"prevalencia reponderada = {(prop['w']*prop['target']).sum()/prop['w'].sum():.4f}  |",
      f"p_real: media={prop['p_real'].mean():.4f} p90={prop['p_real'].quantile(0.9):.4f} max={prop['p_real'].max():.4f}")

def roi_produccion(escenario):
    cm = coste_marginal(escenario, prop['cpl'])
    ben = (prop['w']*prop['target']*prop['cpl']).sum()              # reweighted benefit (clicks only)
    coste_bd = (prop['w']*cm).sum()
    roi_bd = (ben-coste_bd)/coste_bd
    mask = prop['p_real']*prop['cpl'] >= cm                          # decision with REALISTIC prob
    ben_m = (prop.loc[mask,'w']*prop.loc[mask,'target']*prop.loc[mask,'cpl']).sum()
    coste_m = (prop.loc[mask,'w']*cm[mask]).sum()
    roi_m = (ben_m-coste_m)/coste_m if coste_m>0 else float('nan')
    pct = 100*prop.loc[mask,'w'].sum()/prop['w'].sum()
    return {'escenario': escenario, 'ROI_BD_%': round(100*roi_bd),
            'ROI_modelo_%': round(100*roi_m) if not np.isnan(roi_m) else None,
            'emails_%': round(pct, 1)}

prod = pd.DataFrame([roi_produccion(e) for e in ESCENARIOS])
print('=== ROI a escala de PRODUCCION (prior real 2%) ===')
print(prod.to_string(index=False))
print('-> A 2% real, enviar a todos pierde dinero siempre; ni el targeting rescata el canal:')
print('   el mejor caso (coste fijo bajo) queda en break-even (~0%); domina la economia unitaria.')

## 4 · Visualisation (variable cost 30%): ROI by product, BD vs model

In [ ]:
d = tablas['variable_30'].sort_values('ROI_BD_%')
y = np.arange(len(d)); h = 0.4
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(y + h/2, d['ROI_BD_%'], h, label='BD (enviar a todos)', color='#bdbdbd')
ax.barh(y - h/2, d['ROI_modelo_%'], h, label='Modelo (priorizar)', color='#d1495b')
ax.axvline(0, color='gray', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(d['producto'], fontsize=8)
ax.set_xlabel('ROI (%)'); ax.set_title('ROI por producto (coste variable 30%): BD vs modelo')
ax.legend()
plt.tight_layout(); plt.show()

## 5 · Conclusions

With all the **per-email** cost structures, prioritising with the model **improves the overall ROI** compared with sending to everyone (BD). The sending rule is `p_model × CPL ≥ marginal_cost`.

**Overall summary (ROI BD → ROI model):**

| Cost structure | ROI BD | ROI model | Emails sent | Clicks retained | Products that improve |
|---|---|---|---|---|---|
| Fixed (€1/email) | 101% | **322%** | 13,2% | 23,5% | 19/25 |
| Variable 20% CPL | 18% | **143%** | 2,9% | 5,9% | 21/25 |
| Variable 30% CPL | −21% | **87%** | 0,5% | 1,1% | 16/25 |
| Variable 40% CPL | −41% | **102%** | 0,1% | 0,4% | 11/25 |
| Mixed (€0,50 + 15% CPL) | 13% | **132%** | 2,3% | 4,8% | 21/25 |

**Readings:**
- **Fixed (€1):** the model triples the ROI (101% → 322%) by sending only to the 13% with the highest propensity. It is the scenario with the best ROI/volume balance (it retains ~24% of the clicks).
- **Variable (% of the CPL):** the rule reduces to `p_model ≥ %`. The higher the percentage, the more demanding the threshold and the less is sent. Sending to everyone (BD) is already loss-making at 30–40% (negative ROI), because the cost per email is high relative to the real click rate (~24%); the model rescues it to positive, but at 30–40% it sends so little (<1%) that the business is token. At 20% lies the reasonable point (ROI 143% and still capturing ~6% of the clicks).
- **Mixed (€0,50 + 15% CPL):** intermediate; the model multiplies the ROI by ~10 (13% → 132%) by sending to 2,3%.

**Trade-off:** the model raises the ROI by drastically reducing volume (it discards the low-propensity clicks). The very high ROI in variable 30–40% is achieved on very few sends: if the aim is to maximise total clicks rather than efficiency per euro, a low percentage (or a fixed cost) with a less demanding threshold is preferable. The choice depends on the business objective: efficiency (€ per click) versus reach (total clicks).

> Note: CPL imputed to the median (€8) in the records with no CPL reported; the fixed cost, the variable percentages and the mixed combination are editable parameters in section 2.

## 6 · Note on the real prior

**Two readings** of this notebook's ROI are distinguished:

- **ROI on the balanced sample** (sections 3-5): compares, *in relative terms*, "send to
  everyone" versus "prioritise with the model", using `p_model` (training scale). The **ordering**
  of the products by ROI does not depend on the real prior, because the prior correction is monotonic
  (it changes the level of the probabilities, not their order).
- **ROI at production scale** (section 3.bis): reweights the sample to the real prior (~2%) and decides
  using the corrected probability `p_real`. Here the **level** of the ROI in euros does depend on the prior, and at
  a real rate of 2% the picture is much more demanding: sending to everyone loses money in all the
  structures and the model only rescues the low fixed cost to positive (€1 → +14%).

In summary: what is **invariant** to the prior is the *ranking* of products by ROI; the absolute *level*
of the ROI does change with the prior, and that is why it is analysed explicitly in section 3.bis.